In [91]:
%pip -q install google-genai

In [92]:
# Configura a API Key do Google Gemini

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [93]:
# Instalar Framework de agentes do Google ################################################
!pip install -q google-adk

In [94]:
# Configura o cliente da SDK do Gemini

from google import genai

client = genai.Client()

MODEL_ID = "gemini-2.0-flash"

In [95]:
# Pergunta ao Gemini uma informação mais recente que seu conhecimento

from IPython.display import HTML, Markdown

# Perguntar pro modelo quando é a próxima imersão de IA ###############################################
resposta = client.models.generate_content(
    model=MODEL_ID,
    contents='Quando é a próxima Imersão IA com Google Gemini da Alura?',
)

# Exibe a resposta na tela
display(Markdown(f"Resposta:\n {resposta.text}"))

Resposta:
 A Alura não tem uma data fixa para a Imersão IA com Google Gemini. A melhor forma de saber quando a próxima edição será realizada é:

*   **Acessar a página da Imersão:** Procure no site da Alura pela página específica da Imersão IA com Google Gemini. Geralmente, as datas das próximas edições são divulgadas lá.
*   **Acompanhar as redes sociais da Alura:** Siga a Alura no Instagram, LinkedIn, Twitter e outras plataformas. Eles costumam anunciar novos cursos e imersões por lá.
*   **Assinar a newsletter da Alura:** Cadastre seu e-mail na newsletter da Alura para receber informações sobre lançamentos e eventos em primeira mão.

In [96]:
# Pergunta ao Gemini uma informação utilizando a busca do Google como contexto

response = client.models.generate_content(
    model=MODEL_ID,
    contents='Quando é a próxima Imersão IA com Google Gemini da Alura?',
    config={"tools": [{"google_search": {}}]}
)

# Exibe a resposta na tela
display(Markdown(f"Resposta:\n {response.text}"))

Resposta:
 A próxima Imersão IA com Google Gemini da Alura acontecerá de 12 a 16 de maio de 2025. As inscrições estarão abertas até o dia 11 de maio de 2025.

In [97]:
# Exibe a busca
print(f"Busca realizada: {response.candidates[0].grounding_metadata.web_search_queries}")
# Exibe as URLs nas quais ele se baseou
print(f"Páginas utilizadas na resposta: {', '.join([site.web.title for site in response.candidates[0].grounding_metadata.grounding_chunks])}")
print()
display(HTML(response.candidates[0].grounding_metadata.search_entry_point.rendered_content))

Busca realizada: ['Alura Imersão IA com Google Gemini próxima edição']
Páginas utilizadas na resposta: thallesbenicio.com.br



In [98]:
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types  # Para criar conteúdos (Content e Part)
from datetime import date
import textwrap # Para formatar melhor a saída de texto
from IPython.display import display, Markdown # Para exibir texto formatado no Colab
import requests # Para fazer requisições HTTP
import warnings

warnings.filterwarnings("ignore")

In [99]:
# Função auxiliar que envia uma mensagem para um agente via Runner e retorna a resposta final
def call_agent(agent: Agent, message_text: str) -> str:
    # Cria um serviço de sessão em memória
    session_service = InMemorySessionService()
    # Cria uma nova sessão (você pode personalizar os IDs conforme necessário)
    session = session_service.create_session(app_name=agent.name, user_id="user1", session_id="session1")
    # Cria um Runner para o agente
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
    # Cria o conteúdo da mensagem de entrada
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""
    # Itera assincronamente pelos eventos retornados durante a execução do agente
    for event in runner.run(user_id="user1", session_id="session1", new_message=content):
        if event.is_final_response():
          for part in event.content.parts:
            if part.text is not None:
              final_response += part.text
              final_response += "\n"
    return final_response

In [100]:
# Função auxiliar para exibir texto formatado em Markdown no Colab
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [101]:
####################################################################################
# --- Agente 1: Responsável por formatar o historico informado --- #
####################################################################################
def agente_formata_historico(historico):

    agente_formata_historico = Agent(
        name="agente_formata_historico",
        model="gemini-2.0-flash",
        # Inserir as instruções do Agente agente_planejador #################################################
        instruction="""
        Sua função é receber dados históricos de quantos Km um veículo rodou em alguns meses que o usuário irá
        informar e escrever o output neste formato: janeiro, 31 dias, 1112 Km; fevereiro, 28 dias, 580 Km; etc.
        Este output deve bater com os meses e quilometragem que o usuário informar. Você não deve responder nada
        além disso. A quantidade de dias que você vai informar junto com o mês e quilometragem é a quantidade de
        dias do mês.
        """,
        description="Agente responsável por formatar o histórico"
    )

    entrada_agente_agente_formata_historico = f": {historico}/n"

    saida_agente_formata_historico = call_agent(agente_formata_historico, entrada_agente_agente_formata_historico)
    return saida_agente_formata_historico

In [102]:
####################################################################################
# --- Agente 2: Responsável por calcular a média de Km rodados em cada mês --- #
####################################################################################
def agente_calcula_Km_medio_por_mes(historico):

    calculador_media_mensal = Agent(
        name="agente_calcula_Km_medio_por_mes",
        model="gemini-2.0-flash",
        # Inserir as instruções do Agente agente_planejador #################################################
        instruction="""
        Sua função é receber dados históricos formatados e calcular, para cada mês presente, quantos Km foram rodados.
        Exemplo:
        Input: janeiro, 31 dias, 1112 Km; fevereiro, 28 dias, 580 Km
        Cálculo: janeiro 1112/31, fevereiro 580/28
        Output: janeiro 35,87 Km; fevereiro 20,71 Km;
        """,
        description="Agente responsável por calcular a média de Km rodados em cada mês"
    )

    entrada_agente_calcula_Km_medio_por_mes = f": {historico}/n"

    saida_agente_calcula_Km_medio_por_mes = call_agent(calculador_media_mensal, entrada_agente_calcula_Km_medio_por_mes)
    return saida_agente_calcula_Km_medio_por_mes

In [103]:
##########################################################################################
# --- Agente 3: Responsável por fazer o cálculo estatístico básico --- #
##########################################################################################
def agente_estatistico(historico):
    agente_estatistico = Agent(
        name="agente_estatistico",
        model="gemini-2.0-flash",
        # Inserir as instruções do Agente Planejador #################################################
        instruction="""
        Você deve calcular a média -1 desvio padrão e a média +1 desvio padrão.
        Apresente a saída neste formato: média mínima 7,60; média máxima 10,50.
        Não escreva nada além disso.
        """,
        description="Agente que faz o cálculo estatístico básico"
    )

    entrada_do_agente_estatistico = f"{historico}\n"
    # Executa o agente
    saida_do_agente_estatistico = call_agent(agente_estatistico, entrada_do_agente_estatistico)
    return saida_do_agente_estatistico

In [104]:
####################################################################################
# --- Agente 4: Agente responsável por retornar o Km da próxima revisão --- #
####################################################################################
def agente_km_proxima_revisao(odometro_atual):
    agente_km_proxima_revisao = Agent(
        name="agente_km_proxima_revisao",
        model="gemini-2.0-flash",
        instruction="""
            Sua resposta deve ser apenas o número seguido de "Km".
            """,
        description="Agente responsável por retornar o Km da próxima revisão."
    )
    entrada_do_agente_km_proxima_revisao = f"O odômetro do meu carro mostra {odometro_atual}. Qual é a próxima quilometragem divisível por 10 mil?"
    # Executa o agente
    saida_do_agente_km_proxima_revisao = call_agent(agente_km_proxima_revisao, entrada_do_agente_km_proxima_revisao)
    return saida_do_agente_km_proxima_revisao

In [119]:
####################################################################################
# --- Agente 5: Agente responsável por estimar o mês da próxima revisão --- #
####################################################################################
def agente_estimativa_mes_proxima_revisao(km_proxima_revisao, limites, odometro_atual, data_de_hoje):
    agente_estimativa_mes_proxima_revisao = Agent(
        name="agente_correcao_da_qtde_dias_do_mes",
        model="gemini-2.0-flash",
        instruction="""
              Responda de forma bem sucinta ao prompt, entregando apenas 'Você deve planejar fazer a próxima revisão entre ' e o resultado final.
            """,
        description="Agente responsável por estimar o mês da próxima revisão."
    )
    entrada_do_agente_correcao_da_qtde_dias_do_mes = f""": A próxima revisão é quando o odometro do meu veículo chegar em {km_proxima_revisao}\n.
    Considerando a média mínima e a média máxima por dia, em qual mês meu carro vai chegar em na quilometragem da próxima revisão?
    Considere essas as médias mínima e máxima: {limites}.\n
    O odômetro do meu veículo está com esta quilometragem: {odometro_atual}.\n
    Considere que a data atual é {data_de_hoje}.
    """
    # Executa o agente
    saida_do_agente_correcao_da_qtde_dias_do_mes = call_agent(agente_estimativa_mes_proxima_revisao,entrada_do_agente_correcao_da_qtde_dias_do_mes)
    return saida_do_agente_correcao_da_qtde_dias_do_mes

In [118]:
print("Iniciando o Sistema de Previsão de Manutenção Veicular")
print("######################################################")
print("")

# --- Pedir para o usuário inserir o histórico dos últimos meses de Km rodados ---
print("🚗 Por favor, digite a quilometragem rodada com seu veículo nos últimos meses. Recomendo informar pelo menos 12 meses, mas pode ser o histórico que você tiver.")
print("---")
print("💡 Ideia: Você pode pegar o histórico usando as estatísticas da sua Timeline no Google Maps ;)")
print("---")
historico = input("➡️ Insira o histórico seguindo este formato: Abril 541 Km, Maio 657 Km, Junho 800 Km, etc. ")
print("")
print("")
odometro_atual = input("➡️ E aqui, informe a quilometragem atual do seu veículo: ")
print("")
print("")
data_de_hoje = date.today().strftime("%d/%m/%Y")

# Inserir lógica do sistema de agentes ################################################
if not historico:
    print("Você precisa digitar a quilometragem histórica mensal.")
else:
  if not odometro_atual:
    print("Você precisa digitar a quantidade de Km no odometro do seu veículo.")
  else:
    print(f"Maravilha! Iniciar o cálculo de previsão!")
    print("##########################################")

    historico_formatado = agente_formata_historico(historico)
    #print("\n--- 📝 Resultado do Agente 1 ---\n")
    #display(to_markdown(historico_formatado))
    #print("--------------------------------------------------------------")

    media_Km_por_mes = agente_calcula_Km_medio_por_mes(historico_formatado)
    #print("\n--- 📝 Resultado do Agente 2 ---\n")
    #display(to_markdown(media_Km_por_mes))
    #print("--------------------------------------------------------------")

    estatistico = agente_estatistico(media_Km_por_mes)
    #print("\n--- 📝 Resultado do Agente 3 ---\n")
    #display(to_markdown(estatistico))
    #print("--------------------------------------------------------------")

    km_proxima_revisao = agente_km_proxima_revisao(odometro_atual)
    #print("\n--- 📝 Resultado do Agente 4 ---\n")
    #display(to_markdown(km_proxima_revisao))
    #print("--------------------------------------------------------------")

    estimativa_mes_proxima_revisao = agente_estimativa_mes_proxima_revisao(km_proxima_revisao, estatistico, odometro_atual, data_de_hoje)
    #print("\n--- 📝 Resultado do Agente 5 ---\n")
    display(to_markdown(estimativa_mes_proxima_revisao))
    print("--------------------------------------------------------------")

Iniciando o Sistema de Previsão de Manutenção Veicular
######################################################

🚗 Por favor, digite a quilometragem rodada com seu veículo nos últimos meses. Recomendo informar pelo menos 12 meses, mas pode ser o histórico que você tiver.
---
💡 Ideia: Você pode pegar o histórico usando as estatísticas da sua Timeline no Google Maps ;)
---
➡️ Insira o histórico seguindo este formato: Abril 541 Km, Maio 657 Km, Junho 800 Km, etc. outubro 1112 Km; novembro 835 Km; dezembro 1173 Km; janeiro 1257 Km; fevereiro 690 Km; março 906 Km; abril 583 Km;


➡️ E aqui, informe a quilometragem atual do seu veículo: 172000


Maravilha! Iniciar o cálculo de previsão!
##########################################

--- 📝 Resultado do Agente 1 ---



> outubro, 31 dias, 1112 Km; novembro, 30 dias, 835 Km; dezembro, 31 dias, 1173 Km; janeiro, 31 dias, 1257 Km; fevereiro, 28 dias, 690 Km; março, 31 dias, 906 Km; abril, 30 dias, 583 Km;


--------------------------------------------------------------

--- 📝 Resultado do Agente 2 ---



> ok. Aqui estão os cálculos e os resultados para cada mês:
> 
> *   **outubro:** 1112 Km / 31 dias = 35,87 Km/dia
> *   **novembro:** 835 Km / 30 dias = 27,83 Km/dia
> *   **dezembro:** 1173 Km / 31 dias = 37,84 Km/dia
> *   **janeiro:** 1257 Km / 31 dias = 40,55 Km/dia
> *   **fevereiro:** 690 Km / 28 dias = 24,64 Km/dia
> *   **março:** 906 Km / 31 dias = 29,23 Km/dia
> *   **abril:** 583 Km / 30 dias = 19,43 Km/dia
> 
> **Output:** outubro 35,87 Km; novembro 27,83 Km; dezembro 37,84 Km; janeiro 40,55 Km; fevereiro 24,64 Km; março 29,23 Km; abril 19,43 Km;


--------------------------------------------------------------

--- 📝 Resultado do Agente 3 ---



...

--------------------------------------------------------------

--- 📝 Resultado do Agente 4 ---



> 180000Km
> 


--------------------------------------------------------------

--- 📝 Resultado do Agente 5 ---



> Você deve planejar fazer a próxima revisão em 25/08/2025 e 09/07/2025.


--------------------------------------------------------------
